In [7]:
from shapely.geometry import Point, Polygon, MultiPolygon, LineString
import folium
import geopandas as gpd
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from map2grid import plot_with_google_satellite

In [10]:
gens_gdf = gpd.read_file('../outputs/plant_update.gpkg')
lines_gdf = gpd.read_file('../outputs/table_lines_200m.gpkg')
subs = gpd.read_file("../data/osm/substation.gpkg", driver='GPKG')
subs = subs.to_crs(epsg=4326)
gens_gdf = gens_gdf.to_crs(epsg=4326)
lines_gdf = lines_gdf.to_crs(epsg=4326)

data_busbars = pd.read_excel('../outputs/data_busbars.xlsx')
data_busbars_gdf = gpd.GeoDataFrame(data_busbars, geometry=gpd.GeoSeries.from_wkt(data_busbars['geometry']))

data_singular_ways = pd.read_excel('../outputs/data_singular_ways.xlsx')
data_singular_ways_gdf = gpd.GeoDataFrame(data_singular_ways, geometry=gpd.GeoSeries.from_wkt(data_singular_ways['geometry']))


m = plot_with_google_satellite(
    data=lines_gdf,
    data_busbars= data_busbars_gdf,
    data_singular_ways=data_singular_ways_gdf,
    xlim=(107.631, 107.640), ylim=(11.964, 11.9722),
    subs_gdf=subs,
    gens_gdf=gens_gdf
    )
m.save('../figures/map_with_nodes.html')

c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(


In [11]:
import folium
import geopandas as gpd
from shapely.geometry import LineString, Point, Polygon, MultiPolygon

def plot_with_google_satellite_and_gpkg(gpkg_path, output_html_path, xlim, ylim):
    # Step 1: 地图中心点
    center_lat = (ylim[0] + ylim[1]) / 2
    center_lon = (xlim[0] + xlim[1]) / 2

    # Step 2: 创建 Folium 地图，使用 Google Satellite 底图
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles=None)
    
    # 添加 Google Satellite 图层
    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google Satellite",
        name="Google Satellite",
        overlay=True,
        control=True
    ).add_to(m)
    
    # Step 3: 读取 GPKG 文件并加载图层
    gdf_nodes = gpd.read_file(gpkg_path, layer='nodes').to_crs(epsg=4326)
    gdf_lines = gpd.read_file(gpkg_path, layer='lines').to_crs(epsg=4326)
    gdf_loads = gpd.read_file(gpkg_path, layer='loads').to_crs(epsg=4326)

    # Step 4: 绘制不同的元素，根据 disconnected flag 设置颜色，并添加 Popup
    def plot_geometry(gdf, layer_name):
        # 先绘制 disconnected == 'no' 的元素
        for _, row in gdf[gdf['disconnected'] == 'no'].iterrows():
            geom = row.geometry
            color = '#a4ec7e'  # disconnected == 'no' 颜色

            # 根据图层类型生成对应的 Popup 内容
            if layer_name == 'nodes':
                popup_content = f"NodeID: {row['NodeID']}<br>OriginalID: {row['OriginalID']}"
            elif layer_name == 'lines':
                popup_content = f"LineID: {row['LineID']}<br>fromNode: {row['fromNode']}<br>toNode: {row['toNode']}<br>osm_id: {row['osm_id']}"
            elif layer_name == 'loads':
                popup_content = f"osm_id: {row['osmid']}<br>BusID: {row['BusID']}"

            popup = folium.Popup(popup_content, max_width=300)

            # 绘制 LineString
            if isinstance(geom, LineString):
                coords = [(lat, lon) for lon, lat in geom.coords]
                folium.PolyLine(coords, color=color, weight=2, opacity=0.7, popup=popup).add_to(m)
            # 绘制 Point
            elif isinstance(geom, Point):
                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=5,
                    color=color,
                    fill=True,
                    fill_opacity=0.7,
                    popup=popup
                ).add_to(m)
            # 绘制 Polygon
            elif isinstance(geom, Polygon):
                coords = list(geom.exterior.coords)
                folium.Polygon(
                    locations=coords,
                    color=color,
                    weight=2,
                    opacity=0.7,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.3,
                    popup=popup
                ).add_to(m)
            # 绘制 MultiPolygon
            elif isinstance(geom, MultiPolygon):
                for poly in geom.geoms:
                    coords = list(poly.exterior.coords)
                    folium.Polygon(
                        locations=coords,
                        color=color,
                        weight=2,
                        opacity=0.7,
                        fill=True,
                        fill_color=color,
                        fill_opacity=0.3,
                        popup=popup
                    ).add_to(m)

        # 然后绘制 disconnected == 'yes' 的元素
        for _, row in gdf[gdf['disconnected'] == 'yes'].iterrows():
            geom = row.geometry
            color = '#ca235d'  # disconnected == 'yes' 颜色

            # 根据图层类型生成对应的 Popup 内容
            if layer_name == 'nodes':
                popup_content = f"NodeID: {row['NodeID']}<br>OriginalID: {row['OriginalID']}"
            elif layer_name == 'lines':
                popup_content = f"LineID: {row['LineID']}<br>fromNode: {row['fromNode']}<br>toNode: {row['toNode']}<br>osm_id: {row['osm_id']}"
            elif layer_name == 'loads':
                popup_content = f"osm_id: {row['osmid']}<br>BusID: {row['BusID']}"

            popup = folium.Popup(popup_content, max_width=300)

            # 绘制 LineString
            if isinstance(geom, LineString):
                coords = [(lat, lon) for lon, lat in geom.coords]
                folium.PolyLine(coords, color=color, weight=2, opacity=0.7, popup=popup).add_to(m)
            # 绘制 Point
            elif isinstance(geom, Point):
                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=5,
                    color=color,
                    fill=True,
                    fill_opacity=0.7,
                    popup=popup
                ).add_to(m)
            # 绘制 Polygon
            elif isinstance(geom, Polygon):
                coords = list(geom.exterior.coords)
                folium.Polygon(
                    locations=coords,
                    color=color,
                    weight=2,
                    opacity=0.7,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.3,
                    popup=popup
                ).add_to(m)
            # 绘制 MultiPolygon
            elif isinstance(geom, MultiPolygon):
                for poly in geom.geoms:
                    coords = list(poly.exterior.coords)
                    folium.Polygon(
                        locations=coords,
                        color=color,
                        weight=2,
                        opacity=0.7,
                        fill=True,
                        fill_color=color,
                        fill_opacity=0.3,
                        popup=popup
                    ).add_to(m)

    # 绘制每个图层的元素
    plot_geometry(gdf_nodes, 'nodes')
    plot_geometry(gdf_lines, 'lines')
    plot_geometry(gdf_loads, 'loads')

    # 添加 Substations
    subs_gdf = gpd.read_file("../data/osm/substation.gpkg", driver='GPKG')
    if subs_gdf is not None:
        subs_gdf = subs_gdf.to_crs(epsg=4326)
        for idx, row in subs_gdf.iterrows():
            geometry = row.geometry
            tooltip = row.get("name", "Substation")
            
            if isinstance(geometry, Point):  # 如果是 Point 类型
                lon, lat = geometry.x, geometry.y
                folium.Marker(
                    location=[lat, lon],
                    popup=f"Sub: {tooltip}",
                    icon=folium.Icon(color='grey', icon='info-sign')
                ).add_to(m)

            elif isinstance(geometry, LineString):
                coords = [(lat, lon) for lon, lat in geometry.coords]
                folium.PolyLine(
                    coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, weight=2, opacity=0.7, tooltip=tooltip
                ).add_to(m)

            elif isinstance(geometry, Polygon):  # 如果是 Polygon 类型
                coords = list(geometry.exterior.coords)
                folium.Polygon(locations=coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, tooltip=tooltip).add_to(m)
                
            elif isinstance(geometry, MultiPolygon):  # 如果是 MultiPolygon 类型
                for poly in geometry.geoms:
                    coords = list(poly.exterior.coords)
                    folium.Polygon(locations=coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, tooltip=tooltip).add_to(m)

    # Step 6: 保存地图为 HTML 文件
    m.save(output_html_path)
    print(f"地图已保存为 {output_html_path}")

# 例子：使用 .gpkg 文件路径并保存为 .html
gpkg_path = "../outputs/disconnected_elements_flagged.gpkg"  # 你的 GPKG 文件路径
output_html_path = "../outputs/disconnected_elements_map.html"  # 输出 HTML 文件路径

plot_with_google_satellite_and_gpkg(gpkg_path, output_html_path,
                                    xlim=(107.631, 107.640), ylim=(11.964, 11.9722))


c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
C:\Users\mye500\AppData\Local\Temp\ipykernel_3368\2756773077.py:161: UserWarning: color argument of Icon should be one of: {'darkpurple', 'white', 'lightgray', 'darkgreen', 'darkblue', 'blue', 'purple', 'lightblue', 'orange', 'beige', 'red', 'lightgreen', 'gray', 'black', 'darkred', 'lightred', 'pink', 'cadetblue', 'green'}.
  icon=folium.Icon(color='grey', icon='info-sign')


地图已保存为 ../outputs/disconnected_elements_map.html


In [12]:
def plot_with_google_satellite_and_gpkg(gpkg_path, output_html_path, xlim, ylim):
    # Step 1: 地图中心点
    center_lat = (ylim[0] + ylim[1]) / 2
    center_lon = (xlim[0] + xlim[1]) / 2

    # Step 2: 创建 Folium 地图，使用 Google Satellite 底图
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=None)
    
    # 添加 Google Satellite 图层
    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google Satellite",
        name="Google Satellite",
        overlay=True,
        control=True
    ).add_to(m)

    # Step 3: 添加 Substations
    subs_gdf = gpd.read_file("../data/osm/substation.gpkg", driver='GPKG')
    if subs_gdf is not None:
        subs_gdf = subs_gdf.to_crs(epsg=4326)
        for idx, row in subs_gdf.iterrows():
            geometry = row.geometry
            tooltip = row.get("name", "Substation")
            
            if isinstance(geometry, Point):  # 如果是 Point 类型
                lon, lat = geometry.x, geometry.y
                folium.Marker(
                    location=[lat, lon],
                    popup=f"Sub: {tooltip}",
                    icon=folium.Icon(color='grey', icon='info-sign')
                ).add_to(m)

            elif isinstance(geometry, LineString):
                coords = [(lat, lon) for lon, lat in geometry.coords]
                folium.PolyLine(
                    coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, weight=2, opacity=0.7, tooltip=tooltip
                ).add_to(m)

            elif isinstance(geometry, Polygon):  # 如果是 Polygon 类型
                coords = list(geometry.exterior.coords)
                folium.Polygon(locations=coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, tooltip=tooltip).add_to(m)
                
            elif isinstance(geometry, MultiPolygon):  # 如果是 MultiPolygon 类型
                for poly in geometry.geoms:
                    coords = list(poly.exterior.coords)
                    folium.Polygon(locations=coords, color='black', fill=True, fill_color='grey', fill_opacity=0.5, tooltip=tooltip).add_to(m)

    # Step 4: 读取 GPKG 中所有图层
    gdf_nodes = gpd.read_file(gpkg_path, layer='nodes').to_crs(epsg=4326)
    gdf_lines = gpd.read_file(gpkg_path, layer='lines').to_crs(epsg=4326)
    gdf_loads = gpd.read_file(gpkg_path, layer='loads').to_crs(epsg=4326)

    # Step 5: 绘制顺序：先绘制 disconnected == 'no'，再绘制 'yes'
    for disc in ['no', 'yes']:
        # --- 绘制 lines ---
        sub_lines = gdf_lines[gdf_lines['disconnected'] == disc]
        for _, row in sub_lines.iterrows():
            geom = row.geometry
            if isinstance(geom, LineString):
                coords = [(lat, lon) for lon, lat in geom.coords]
                color = '#ca235d' if disc == 'yes' else '#a4ec7e'
                popup_html = (
                    f"LineID: {row.get('LineID')}<br>"
                    f"fromNode: {row.get('fromNode')}<br>"
                    f"toNode: {row.get('toNode')}<br>"
                    f"osm_id: {row.get('osm_id')}"
                )
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=2,
                    opacity=0.7,
                    popup=folium.Popup(popup_html, max_width=300)
                ).add_to(m)

        # --- 绘制 nodes ---
        sub_nodes = gdf_nodes[gdf_nodes['disconnected'] == disc]
        for _, row in sub_nodes.iterrows():
            geom = row.geometry
            if isinstance(geom, Point):
                color = '#ca235d' if disc == 'yes' else '#a4ec7e'
                popup_html = (
                    f"NodeID: {row.get('NodeID')}<br>"
                    f"OriginalID: {row.get('OriginalID')}"
                )
                folium.CircleMarker(
                    location=[geom.y, geom.x],
                    radius=5,
                    color=color,
                    fill=True,
                    fill_opacity=0.7,
                    popup=folium.Popup(popup_html, max_width=300)
                ).add_to(m)

        # --- 绘制 loads（polygon/multipolygon）---
        sub_loads = gdf_loads[gdf_loads['disconnected'] == disc]
        for _, row in sub_loads.iterrows():
            color = '#ca235d' if disc == 'yes' else '#a4ec7e'
            popup_html = (
                f"osm_id: {row.get('osmid')}<br>"
                f"BusID: {row.get('BusID')}"
            )
            if isinstance(row.geometry, Polygon):
                geom_list = [row.geometry]
            elif isinstance(row.geometry, MultiPolygon):
                geom_list = row.geometry.geoms
            else:
                continue  # skip if not polygonal

            for poly in geom_list:
                coords = [(lat, lon) for lon, lat in poly.exterior.coords]
                folium.Polygon(
                    locations=coords,
                    color=color,
                    weight=2,
                    opacity=0.7,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.3,
                    popup=folium.Popup(popup_html, max_width=300)
                ).add_to(m)
    
    # Step 6: 保存地图为 HTML 文件
    m.save(output_html_path)

In [15]:
gpkg_path = "../outputs/disconnected_elements_flagged.gpkg"
output_html_path = "../outputs/disconnected_elements_map.html"

plot_with_google_satellite_and_gpkg(
    gpkg_path,
    output_html_path,
    xlim=(107.631, 107.640),
    ylim=(11.964, 11.9722)
)

c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: driver GPKG does not support open option DRIVER
  return ogr_read(
C:\Users\mye500\AppData\Local\Temp\ipykernel_3368\463384177.py:31: UserWarning: color argument of Icon should be one of: {'darkpurple', 'white', 'lightgray', 'darkgreen', 'darkblue', 'blue', 'purple', 'lightblue', 'orange', 'beige', 'red', 'lightgreen', 'gray', 'black', 'darkred', 'lightred', 'pink', 'cadetblue', 'green'}.
  icon=folium.Icon(color='grey', icon='info-sign')
